# Handout 3 - Final Challenge: Platform-only SFT experiment

This is a **working guide and evidence log**, not a notebook that calls the API. Complete every action in the OpenAI Platform, then replace each `[...]` field with your evidence.

**Course minimum:** 10 training examples, 10 validation examples, and 10 held-out test examples; baseline before training; controlled base-versus-fine-tuned comparison; screenshots; and a conclusion.

> Important: do not upload the test file to the fine-tuning job. Do not put the full target format into the inference system message: the examples should teach the detailed behaviour.

## 0. Choose a focused, repeatable behaviour

A good SFT task is a repeated transformation where you can judge a clearly good output. It is **not** mainly a way to add current facts.


| Decision | Your answer |
|---|---|
| Use case / audience | A creative-writing tool for writers who want to transform plain scene notes into polished literary vignettes. |
| Recurring input | Short factual descriptions of scenes, characters, and situations. |
| Exact behaviour to learn | Rewrite the description as a 60–100 word literary vignette using atmospheric sensory detail, restrained emotional implication, and polished prose, while preserving the original events. |
| Measurable base-model failure mode | The base model may be inconsistent in tone, over-explain emotions, invent plot details, or fail to maintain the desired restrained literary style. |
| Why a prompt alone is insufficient | A short prompt can request literary prose, but outputs may vary substantially in tone and stylistic consistency. SFT is being tested as a way to reproduce the same transformation more reliably. |
| What a useful improvement would look like | The fine-tuned model should preserve the source facts while more consistently producing atmospheric, restrained literary prose without additional explanation. |

**Quality gate:** If you cannot write a target answer for a new input without guessing, narrow or change the task.

## 1. Define the prompt contract before writing data

The system message must appear identically in training, validation, baseline testing, and final testing. Keep it compact: define the general job and grounding limits, but avoid revealing the full output template.

**My system message (copy exactly from here into every dataset line and the Playground Prompt field):**

```text
Rewrite the supplied scene description as a short literary vignette. Preserve the information in the description and return only the vignette.
```

**Hidden target contract taught by demonstrations:** `[60–100 words; polished literary register; atmosphere created through sensory or environmental detail; emotion suggested indirectly rather than explicitly explained; original facts preserved; no major invented events; no preamble or explanation.]`

**Inference user-request variants to represent:**

1. Rewrite this scene as a literary vignette:
2. Turn the following description into a short literary passage:
3. Create a literary vignette from this scene:
4. Render this scene as short literary prose:

Use natural variation in phrasing, but keep the underlying task unchanged.

## 2. Design 30 separate examples

Create **three separate JSONL files**, each with at least 10 examples. Keep the test examples untouched until final evaluation. Avoid duplicates and near-duplicates across splits. Include easier and harder cases in every split, and reserve some realistic wording/scenario variation for test.

| Split      | Filename                             | Count | Purpose                                                        | Uploaded to job? |
| ---------- | ------------------------------------ | ----: | -------------------------------------------------------------- | ---------------- |
| Train      | `literary_vignette_train.jsonl`      |    10 | Parameter updates during SFT                                   | Yes              |
| Validation | `literary_vignette_validation.jsonl` |    10 | Monitor validation behaviour during training                   | Yes              |
| Test       | `literary_vignette_test.jsonl`       |    10 | Final controlled comparison between base and fine-tuned models | **No**           |


Plan your coverage before authoring:

| Case type / variation              | Train IDs            | Validation IDs     | Test IDs            |
| ---------------------------------- | -------------------- | ------------------ | ------------------- |
| Typical everyday scenes            | train_01–04          | val_01–04          | test_01–04          |
| Alternative request phrasing       | train_05–07          | val_05–07          | test_05–07          |
| Emotionally implicit situations    | train_03, 06, 08, 10 | val_03, 05, 07, 09 | test_01, 02, 06, 10 |
| Less familiar / held-out scenarios | train_08–10          | val_08–10          | test_08–10          |



Synthetic data generation: The 30 examples were synthetically created with the assistance of ChatGPT according to a predefined target behaviour. Each example pairs a short factual scene description with a manually reviewed gold-standard literary vignette. The outputs were constrained to 60–100 words, preserve the original scene, use atmospheric detail, imply emotion rather than state it directly, maintain a restrained literary register, and avoid major invented facts. The examples were divided into separate training, validation, and held-out test sets, with different scenarios and varied request phrasing to reduce duplication and test generalisation.

All 30 gold-standard responses were reviewed against the same five criteria later used for model evaluation: meaning preservation, atmosphere, implied emotion, literary register, and output discipline.

## 3. Write the JSONL files

JSONL has one complete JSON object per line. Each object contains a messages array with system, user, and assistant roles.

```json
{"messages":[
  {"role":"system","content":"Rewrite the supplied scene description as a short literary vignette. Preserve the information in the description and return only the vignette."},
  {"role":"user","content":"[a natural request plus a plain factual scene description]"},
  {"role":"assistant","content":"[the gold-standard literary vignette]"}
]}
```

**Authoring checklist for every line**

- The system message is identical across all 30 examples.
- The user request varies naturally in phrasing.
- The assistant response follows the agreed target behaviour:
  - 60–100 words
  - preserves the important facts of the scene
  - uses atmospheric sensory or environmental detail
  - implies emotion rather than stating it directly
  - maintains a restrained literary register
  - avoids major invented facts, headings, and explanations
- Each line is valid JSON and contains one messages array.
- Train, validation, and test examples are distinct, with no reused or near-duplicate scenes.

**Synthetic data production**

The dataset was synthetically created with the assistance of ChatGPT according to a predefined target behaviour. Each plain scene description was paired with a gold-standard literary vignette, and all outputs were manually reviewed against the same criteria later used for evaluation: meaning preservation, atmosphere, implied emotion, literary register, and output discipline.

**Manual inspection evidence**

I opened complete examples from all three splits and confirmed that the structure, system message, target behaviour, and separation between train, validation, and test were consistent.

## 4. Baseline in the OpenAI Playground - do this before training

1. Open **Platform > Playground** and select the base model you intend to fine-tune (the handout uses GPT-4.1 nano if it is available to the course organization).
2. Paste the Section 1 system message into the **Prompt** field.
3. Set temperature to `0` if the interface allows it; otherwise record the value. Keep every visible generation setting fixed for both models.
4. Run the same 10 held-out user messages from `literary_vignette_test.jsonl`, once each. Save the raw outputs and a baseline Playground screenshot showing the Prompt field.

| Field | Baseline record |
|---|---|
| Exact base-model ID shown | gpt-4.1-nano-2025-04-14 |
| System message used | Rewrite the supplied scene description as a short literary vignette. Preserve the information in the description and return only the vignette. |
| Temperature / max output tokens / Top P | 0/2048/1 |
| Baseline Playground screenshot filename or location | handout_3_final_challenge_screenshot |

> Do not change models, prompts, requests, or generation settings after this point merely to improve the comparison.

- **Message 1:** Render this scene as short literary prose:\nA woman sits in a parked car outside her childhood home before going inside to help clear it out.
- **Provided Answer:** She switched off the engine but left both hands on the wheel. The house looked smaller from the road than it had in memory, its front hedge uneven, one upstairs curtain half drawn. Cardboard boxes were visible through the open garage. She had come to sort what could be kept and what could be given away. For several minutes she watched the empty doorway, listening to the ticking of the cooling car. Then she gathered the keys from the passenger seat and opened the door.
- **Baseline Output:** She sat quietly in the parked car, its engine softly humming beneath her. Outside, her childhood home loomed silent and still, waiting to be emptied of memories. She took a deep breath, gathering strength, before stepping out into the familiar, bittersweet silence, ready to help clear away the echoes of her past.

- **Message 2:**Create a literary vignette from this scene:\nA man receives a voicemail from his brother after they have not spoken for months. He listens to it twice without replying.
- **Provided Answer:** The message was only forty seconds long. He listened once standing beside the sink, then again after sitting down. His brother's voice sounded ordinary, which somehow made the months between them more noticeable. There was no apology in it, and no accusation either, only a request to call when he had time. The phone screen dimmed in his hand. He pressed replay, stopped himself, and set it face-down on the table. For the rest of the evening, forty seconds seemed to occupy the room.
- **Baseline Output:**He sat alone, the faint glow of his phone casting shadows across his face. The voicemail from his brother played softly, each word lingering in the quiet room. He listened again, the weight of unspoken months pressing between them, before setting the phone down without a reply. Silence settled in, thick and unbroken, as he stared at the screen, caught in the quiet aftermath of a bridge left untraveled.

- **Message 3:** A woman is the first person to arrive at a beach early in the morning. She takes off her shoes but does not enter the water.
- **Provided Answer:** She arrived just as dawn was breaking, the first footprints in the cool, untouched sand. Slipping off her shoes, she felt the soft grains between her toes, grounding her in the quiet serenity of the morning. She stood at the water’s edge, watching the gentle waves lap at the shore, but never stepping in. Instead, she remained there, a solitary figure amid the awakening sea, savoring the stillness before the world stirred.
- **Baseline Output:** The beach was almost without footprints, the sand smoothed overnight by wind and the retreating tide. She left her shoes beside a piece of driftwood and walked to the waterline. Each small wave reached toward her feet and withdrew before touching them. Farther out, the sea was grey-blue and flat beneath the early sky. She stood there long enough for the damp sand to cool her soles. When the sun cleared the buildings behind her, she still had not stepped forward.

- **Message 4:** Turn the following description into a short literary passage:\nA family waits in a hospital corridor while one member is having a routine procedure. Nobody is seriously worried, but conversation keeps stopping.
- **Provided Answer:** They occupied four plastic chairs beneath a television no one was watching. Someone had brought biscuits, still unopened in a paper bag. Their conversation moved easily enough through traffic, parking and the weather, then stopped without warning whenever footsteps approached from the double doors. Each time, they looked up together and then away. The procedure was routine; they had been told this several times. Even so, the corridor divided time into small intervals, each measured by the swing of the doors.
- **Baseline Output:** The family sat huddled in the sterile quiet of the hospital corridor, their chairs forming a loose circle of shared anticipation. The hum of distant activity echoed softly around them, but here, time seemed to slow. They exchanged quiet glances, words lingering on the tip of their tongues, only to fade away into pauses filled with unspoken reassurance. Nobody was truly worried—just the gentle tension of waiting, a fragile pause in the rhythm of routine. Outside the door, their loved one was inside, undergoing what was expected to be a simple procedure, yet the silence between them held all the unspoken hope and quiet patience that words could not quite capture.

- **Message 5:** Render this scene as short literary prose:\nA young woman packs for her first day at a new job and changes her outfit several times.
- **Provided Answer:** By eight o'clock, three blouses lay across the bed and two pairs of shoes had been returned to the wardrobe. She stood before the mirror in the fourth arrangement, smoothing a sleeve that was already smooth. Her bag waited open on the chair, containing a notebook, a pen and nothing that could explain what the day would require. She checked the time and changed her earrings. When she finally closed the bedroom door behind her, the discarded clothes looked unexpectedly certain of themselves.
- **Baseline Output:** In the quiet of her room, she moved with tentative purpose, opening her closet to select the perfect outfit. She tried on one, then another, each change a small act of hope and doubt. The mirror reflected her reflection—an uncertain smile, a flicker of confidence. She adjusted a collar, swapped shoes, and hesitated before finally settling on something that felt just right. As she zipped her bag and took a deep breath, she carried with her the quiet anticipation of a new beginning, dressed in the many versions of herself she had tried on today.

- **Message 6:** Create a literary vignette from this scene:\nAn older woman hears music from a neighbour's garden while folding laundry alone in her bedroom.
- **Provided Answer:** She was folding towels when the music began, faint through the open window and interrupted by the sound of traffic. Someone next door had chosen a song she knew immediately, though she could not have said when she last heard it. She held one towel by its corners without bringing them together. In the garden below, sheets moved slightly on the line. The song continued, cheerful and distant. By the time she resumed folding, she had missed the middle verse and remembered an entire summer.
- **Baseline Output:** She stood by the window, folding laundry in the quiet of her room, when a faint melody drifted through the open air. It was a song she hadn’t heard in years—soft, lilting, carrying memories like whispers. The music seeped through the garden fence, weaving itself into the stillness of her solitude. She paused, her hands resting on the fabric, eyes closed for a moment as the notes stirred something long buried. Outside, the world continued unaware, but inside her, the song lingered—an echo of days gone by, gentle and persistent, filling the empty spaces with a quiet, tender warmth.

- **Message 7:** Rewrite this scene as a literary vignette:\nA man returns a library book several weeks late. He has carried it in his bag every day but kept forgetting to return it.
- **Provided Answer:** The book emerged from his bag with softened corners and a receipt folded between its pages. For weeks it had travelled with him to work, to cafés, across the city and back again, accumulating intention without progress. At the library desk he placed it beneath the small red scanner and waited. The librarian mentioned the overdue notice mildly. He apologised with more seriousness than the sum required. When the book disappeared onto the returns trolley, his bag felt strangely light, almost undeservedly so.
- **Baseline Output:** He arrived at the library, the weight of the overdue book still tucked inside his bag, its presence a quiet reminder of days lost. For weeks, he had carried it everywhere—through mornings and evenings, in the rush of life—each day promising to return it, yet somehow forgetting. Now, standing at the counter, he hesitated, the overdue fine a small, persistent shadow. He handed over the book, feeling the weight of unspoken apologies, and watched as the librarian’s eyes flicked over the date. A silent acknowledgment passed between them—an unspoken apology for the delay, a quiet acceptance of the simple, human tendency to forget.

- **Message 8:** Turn the following description into a short literary passage:\nTwo friends sit on a wall after a party has ended. They are too tired to call a taxi yet.
- **Provided Answer:** The music had stopped behind them, though a bass note seemed to remain in the pavement. They sat on the low wall with their coats buttoned badly, sharing the last of a bottle of water. Across the road, cleaners were already moving chairs inside a café. One of them opened the taxi app, looked at the price, and closed it again. Neither suggested walking. Dawn had not quite begun, but the night had lost its authority. They stayed where they were, saying very little.
- **Baseline Output:** They settled onto the old stone wall, the night still humming softly around them. The party had long since faded into quiet echoes, and now only the gentle weight of exhaustion pressed upon their shoulders. Neither moved to call a taxi; the effort felt too great, the silence too comfortable. They sat in stillness, watching the darkened street, their words sparse, as if the night itself was holding its breath. In that quiet pause, beneath the fading glow of streetlights, they simply existed—tired, content, and waiting for the dawn to carry them home.

- **Message 9:** Render this scene as short literary prose:\nA teacher stays in an empty classroom after the last lesson, looking at drawings the children have left on the wall.
- **Provided Answer:** After the final bell, the classroom changed quickly. Chairs stood crooked beneath the tables, pencil shavings brightened the floor, and the children's voices retreated along the corridor. She remained by the wall where their drawings had been pinned in two uneven rows. Houses leaned, suns occupied impossible corners, families smiled with identical arms. One picture had begun to curl away from its tack. She pressed it flat with her palm, then gathered her papers slowly, listening to the building settle around her.
- **Baseline Output:** The classroom was silent now, the last lesson long over. She lingered, her gaze drifting across the walls where children's drawings clung like quiet whispers of their imaginations. Crayon strokes and crumpled paper, bright with innocence and wonder, told stories only they knew. She reached out, fingers brushing a faded scribble, feeling the gentle weight of their small worlds left behind. In the stillness, she stayed a moment longer, holding onto the fleeting magic of their childhood, before turning away to leave the quiet room behind.

- **Message 10:** Create a literary vignette from this scene:\nA woman buys flowers for herself on the way home after a difficult day at work.
- **Provided Answer:** She chose the flowers without much thought: a small bunch wrapped in clear plastic, yellow and white among the vegetables and newspapers near the till. On the bus home she held them upright between her knees so the stems would not bend. The day had left a heaviness behind her eyes, though nothing dramatic had happened. In her kitchen she found a jar because the vase was dusty. Once the flowers were in water, the room looked only slightly different. She noticed the difference anyway.
- **Baseline Output:** She paused at the flower stall, the scent of fresh blooms wrapping around her like a quiet reassurance. With deliberate care, she chose a bouquet—vivid petals and delicate greens—something to remind her of beauty amid the chaos. As she paid, her fingers brushed the cool paper wrapping, a small act of kindness she granted herself. Walking home, the flowers nestled in her arms, she felt a subtle lift, a gentle reminder that even on the hardest days, there was still a moment worth savoring—something simple, yet enough to make her smile.

## 5. Create the SFT job in the Platform

Go to **Platform > Fine-tuning > Create**. The interface can change; use the labels that correspond to these choices:

| Field | First-run choice | Record after creation |
|---|---|---|
| Method | Supervised | [...] |
| Base model | Same exact baseline model | [...] |
| Suffix | `literary-vignette-sft` | [...] |
| Seed | `42` (or record platform value) | [...] |
| Training data | `literary_vignette_train.jsonl` only | [...] |
| Validation data | `literary_vignette_validation.jsonl` | [...] |
| Batch size | Auto | [...] |
| Learning-rate multiplier | Auto | [...] |
| Number of epochs | Auto | [...] |

Before clicking Create, capture the configuration screenshot and verify that the test file is not listed. **Do not start a second job yet:** interpret the first run before changing hyperparameters.

## 6. Monitor and interpret the job

Record the job ID, status, selected model, final hyperparameters, and at least one checkpoint. Falling training loss means closer agreement with the training targets. Validation loss reflects behaviour on examples not used for weight updates. Token accuracy is a token-level training diagnostic, not the final task metric. Low loss or high token accuracy alone does **not** prove that the literary-vignette use case is successful.

| Evidence | Record |
|---|---|
| Job ID and final status | ftjob-2X1BWx0Xr8GCNKliOFtO79J7 |
| Output model ID (`ft:...`) | ft:gpt-4.1-nano-2025-04-14:cocreate-learning-labs:literary-vignette-sft:EMtuVMIl |
| Final batch size / LR multiplier / epochs | 3/0.1/1 |
| Checkpoint considered and reason | Checkpoint 112 was considered because validation loss had already converged near zero and validation token accuracy had reached 1.0. Later checkpoints at 224 and 336 showed no visible improvement, so the earliest equally strong checkpoint was preferred to avoid unnecessary additional fitting. |
| Training metrics / dashboard screenshot location | [...] |
| Training loss | 0 |
| Validation loss | 0 |
| Token accuracy | Approximately 1.00 from the first checkpoint onward. |
| Validation behaviour observed | Validation metrics converged early and remained stable; no visible divergence from training metrics was observed. |

**Your short metrics interpretation:**

The loss curves decreased sharply during the first approximately 30–50 training steps and then remained close to zero. Accuracy increased from around 0.7–0.8 to 1.0 and stayed at that level at checkpoints 112, 224, and 336. Since later checkpoints showed no visible validation improvement, checkpoint 112 was selected as the earliest equally strong model. However, these results are diagnostic only: the very small synthetic training and validation sets may make the metrics optimistic, so the held-out test comparison is needed to judge whether the model is genuinely useful.


## 7. Controlled final comparison: baseline model and checkpoint 112 model: 

**Model name:** ft:gpt-4.1-nano-2025-04-14:cocreate-learning-labs:literary-vignette-sft:EMtuT87j:ckpt-step-112

In the Playground, compare the original base model and `ft:...` model side by side. Use the **same** Section 1 Prompt, exact test user message, and generation settings for each row. Run all 10 held-out prompts. Save at least one side-by-side screenshot.

Score every base-model and fine-tuned output using the same five binary criteria (1 point each):

1. **Meaning preserved:** important facts and relationships in the original scene are preserved.
2. **Atmosphere present:** sensory, environmental, spatial, or physical detail creates atmosphere rather than merely restating the scene.
3. **Emotion implied:** emotion is mainly conveyed indirectly through action, gesture, setting, perception, or physical detail rather than explicitly named.
4. **Literary register:** prose is polished, restrained, and stylistically coherent, without melodrama, cliches, conversational explanation, or generic filler.
5. **Output discipline:** the response contains only the vignette, is approximately 60-100 words, and introduces no major unsupported plot facts.

Each response receives a score from 0-5. Evaluate all 10 held-out prompts from `literary_vignette_test.jsonl`; maximum total per model is 50.

| Test ID | Base score /5 | Fine-tuned score /5 | Main difference |
|---|---:|---:|---|
| test_01 | 2 | 4 | The checkpoint is more restrained, but both outputs are below 60 words. |
| test_02 | 4 | 5 | The checkpoint is slightly more restrained; both preserve the main event. |
| test_03 | 5 | 3 | The base output is more scene-specific and meets length; the checkpoint is too short and generic. |
| test_04 | 2 | 2 | Neither is sufficiently restrained; the checkpoint also invents the door opening. |
| test_05 | 3 | 3 | Both preserve the facts but explicitly name emotion and use generic phrasing. |
| test_06 | 4 | 3 | The base output implies memory more effectively; the checkpoint explicitly names longing and adds an unsupported ending. |
| test_07 | 3 | 3 | The checkpoint meets length but is more generic and explicitly states relief; the base output is over length. |
| test_08 | 3 | 3 | Both preserve the scene but neither consistently implies emotion or avoids cliches. |
| test_09 | 4 | 4 | Both preserve the facts; there is no meaningful stylistic improvement. |
| test_10 | 3 | 3 | Both preserve the scene; the checkpoint adds unsupported trembling and explicit emotional framing. |
| **Total** | **33 / 50** | **33 / 50** | **No overall score improvement from the selected checkpoint.** |

Also retain the raw base and fine-tuned outputs. A score without the outputs is hard to audit.

- **Message 1:** Render this scene as short literary prose:\nA woman sits in a parked car outside her childhood home before going inside to help clear it out.
- **Provided Answer:** She switched off the engine but left both hands on the wheel. The house looked smaller from the road than it had in memory, its front hedge uneven, one upstairs curtain half drawn. Cardboard boxes were visible through the open garage. She had come to sort what could be kept and what could be given away. For several minutes she watched the empty doorway, listening to the ticking of the cooling car. Then she gathered the keys from the passenger seat and opened the door.
- **Baseline Output:** She sat quietly in the parked car, its engine softly humming beneath her. Outside, her childhood home loomed silent and still, waiting to be emptied of memories. She took a deep breath, gathering strength, before stepping out into the familiar, bittersweet silence, ready to help clear away the echoes of her past.
- **Fine-tuned:** The woman sat quietly in the parked car, its engine idling softly in the quiet street. Outside, her childhood home stood silent and waiting, its windows reflecting the fading light of day. Soon, she would step out and cross the threshold once more, her hands ready to clear out the memories stored within those walls.
---
- **Message 2:** Create a literary vignette from this scene:\nA man receives a voicemail from his brother after they have not spoken for months. He listens to it twice without replying.
- **Provided Answer:** The message was only forty seconds long. He listened once standing beside the sink, then again after sitting down. His brother's voice sounded ordinary, which somehow made the months between them more noticeable. There was no apology in it, and no accusation either, only a request to call when he had time. The phone screen dimmed in his hand. He pressed replay, stopped himself, and set it face-down on the table. For the rest of the evening, forty seconds seemed to occupy the room.
- **Baseline Output:**He sat alone, the faint glow of his phone casting shadows across his face. The voicemail from his brother played softly, each word lingering in the quiet room. He listened again, the weight of unspoken months pressing between them, before setting the phone down without a reply. Silence settled in, thick and unbroken, as he stared at the screen, caught in the quiet aftermath of a bridge left untraveled.
- **Fine-tuned:** The man sat alone with the faint hum of the room around him, eyes fixed on the small screen of his phone. The voicemail from his brother played once, then again, each time leaving a quiet echo in the space between them. He listened without replying, the silence stretching long after the second playback, as if the words had already settled somewhere deep inside him.
---
- **Message 3:** A woman is the first person to arrive at a beach early in the morning. She takes off her shoes but does not enter the water.
- **Provided Answer:** She arrived just as dawn was breaking, the first footprints in the cool, untouched sand. Slipping off her shoes, she felt the soft grains between her toes, grounding her in the quiet serenity of the morning. She stood at the water’s edge, watching the gentle waves lap at the shore, but never stepping in. Instead, she remained there, a solitary figure amid the awakening sea, savoring the stillness before the world stirred.
- **Baseline Output:** The beach was almost without footprints, the sand smoothed overnight by wind and the retreating tide. She left her shoes beside a piece of driftwood and walked to the waterline. Each small wave reached toward her feet and withdrew before touching them. Farther out, the sea was grey-blue and flat beneath the early sky. She stood there long enough for the damp sand to cool her soles. When the sun cleared the buildings behind her, she still had not stepped forward.
- **Fine-tuned:** The woman arrived at the beach just as dawn was breaking, her footsteps soft on the cool sand. She took off her shoes, feeling the grains slip through her fingers, but she did not step into the water. Instead, she stood there quietly, watching the horizon lighten, her presence a solitary whisper in the vast, waking world.
---
- **Message 4:** Turn the following description into a short literary passage:\nA family waits in a hospital corridor while one member is having a routine procedure. Nobody is seriously worried, but conversation keeps stopping.
- **Provided Answer:** They occupied four plastic chairs beneath a television no one was watching. Someone had brought biscuits, still unopened in a paper bag. Their conversation moved easily enough through traffic, parking and the weather, then stopped without warning whenever footsteps approached from the double doors. Each time, they looked up together and then away. The procedure was routine; they had been told this several times. Even so, the corridor divided time into small intervals, each measured by the swing of the doors.
- **Baseline Output:** The family sat huddled in the sterile quiet of the hospital corridor, their chairs forming a loose circle of shared anticipation. The hum of distant activity echoed softly around them, but here, time seemed to slow. They exchanged quiet glances, words lingering on the tip of their tongues, only to fade away into pauses filled with unspoken reassurance. Nobody was truly worried—just the gentle tension of waiting, a fragile pause in the rhythm of routine. Outside the door, their loved one was inside, undergoing what was expected to be a simple procedure, yet the silence between them held all the unspoken hope and quiet patience that words could not quite capture.
- **Fine-tuned:** The family sat huddled in the sterile quiet of the hospital corridor, each face turned toward the door through which their loved one would soon emerge. Nobody was seriously worried; the routine procedure was just that—routine. Yet conversation kept stopping, words dissolving into the still air, as if the waiting itself needed a moment to breathe. They exchanged glances and small smiles, holding onto the calm that was only just enough, until the door finally opened and the next chapter could begin.
---
- **Message 5:** Render this scene as short literary prose:\nA young woman packs for her first day at a new job and changes her outfit several times.
- **Provided Answer:** By eight o'clock, three blouses lay across the bed and two pairs of shoes had been returned to the wardrobe. She stood before the mirror in the fourth arrangement, smoothing a sleeve that was already smooth. Her bag waited open on the chair, containing a notebook, a pen and nothing that could explain what the day would require. She checked the time and changed her earrings. When she finally closed the bedroom door behind her, the discarded clothes looked unexpectedly certain of themselves.
- **Baseline Output:** In the quiet of her room, she moved with tentative purpose, opening her closet to select the perfect outfit. She tried on one, then another, each change a small act of hope and doubt. The mirror reflected her reflection—an uncertain smile, a flicker of confidence. She adjusted a collar, swapped shoes, and hesitated before finally settling on something that felt just right. As she zipped her bag and took a deep breath, she carried with her the quiet anticipation of a new beginning, dressed in the many versions of herself she had tried on today.
- **Fine-tuned:** The young woman moved with quiet urgency, her hands busy with the task of packing for her first day at a new job. She changed her outfit several times, each choice a small act of hope and doubt, as if the right clothes could somehow make the future clearer. Outside, the world waited, but inside she was caught in a delicate dance of preparation, trying to find the one look that would carry her through the unknown.
---
- **Message 6:** Create a literary vignette from this scene:\nAn older woman hears music from a neighbour's garden while folding laundry alone in her bedroom.
- **Provided Answer:** She was folding towels when the music began, faint through the open window and interrupted by the sound of traffic. Someone next door had chosen a song she knew immediately, though she could not have said when she last heard it. She held one towel by its corners without bringing them together. In the garden below, sheets moved slightly on the line. The song continued, cheerful and distant. By the time she resumed folding, she had missed the middle verse and remembered an entire summer.
- **Baseline Output:** She stood by the window, folding laundry in the quiet of her room, when a faint melody drifted through the open air. It was a song she hadn’t heard in years—soft, lilting, carrying memories like whispers. The music seeped through the garden fence, weaving itself into the stillness of her solitude. She paused, her hands resting on the fabric, eyes closed for a moment as the notes stirred something long buried. Outside, the world continued unaware, but inside her, the song lingered—an echo of days gone by, gentle and persistent, filling the empty spaces with a quiet, tender warmth.
- **Fine-tuned:** The older woman stood alone in her bedroom, the soft rhythm of music drifting through the open window from the neighbor’s garden. She paused, her hands stilling on the fabric, and listened as the melodies wove through the air, gentle and persistent. Folding laundry, she felt a quiet pull of longing and memory, as if the music carried stories she once knew and now only remembered in fragments. When the last shirt was folded, she turned away, the sound lingering in her mind, a delicate reminder of the world beyond her door.
---
- **Message 7:** Rewrite this scene as a literary vignette:\nA man returns a library book several weeks late. He has carried it in his bag every day but kept forgetting to return it.
- **Provided Answer:** The book emerged from his bag with softened corners and a receipt folded between its pages. For weeks it had travelled with him to work, to cafés, across the city and back again, accumulating intention without progress. At the library desk he placed it beneath the small red scanner and waited. The librarian mentioned the overdue notice mildly. He apologised with more seriousness than the sum required. When the book disappeared onto the returns trolley, his bag felt strangely light, almost undeservedly so.
- **Baseline Output:** He arrived at the library, the weight of the overdue book still tucked inside his bag, its presence a quiet reminder of days lost. For weeks, he had carried it everywhere—through mornings and evenings, in the rush of life—each day promising to return it, yet somehow forgetting. Now, standing at the counter, he hesitated, the overdue fine a small, persistent shadow. He handed over the book, feeling the weight of unspoken apologies, and watched as the librarian’s eyes flicked over the date. A silent acknowledgment passed between them—an unspoken apology for the delay, a quiet acceptance of the simple, human tendency to forget.
- **Fine-tuned:** The man carried the library book in his bag every day, its weight a quiet reminder of promises made and forgotten. Weeks had slipped by, unnoticed until today, when he finally returned to the library and handed over the overdue volume. His fingers brushed the counter as he relinquished it, the book feeling lighter now, freed from the weight of his forgetfulness. Outside, the world moved on, but inside he carried a small, silent apology, and the quiet relief of having once again made things right.
---
- **Message 8:** Turn the following description into a short literary passage:\nTwo friends sit on a wall after a party has ended. They are too tired to call a taxi yet.
- **Provided Answer:** The music had stopped behind them, though a bass note seemed to remain in the pavement. They sat on the low wall with their coats buttoned badly, sharing the last of a bottle of water. Across the road, cleaners were already moving chairs inside a café. One of them opened the taxi app, looked at the price, and closed it again. Neither suggested walking. Dawn had not quite begun, but the night had lost its authority. They stayed where they were, saying very little.
- **Baseline Output:** They settled onto the old stone wall, the night still humming softly around them. The party had long since faded into quiet echoes, and now only the gentle weight of exhaustion pressed upon their shoulders. Neither moved to call a taxi; the effort felt too great, the silence too comfortable. They sat in stillness, watching the darkened street, their words sparse, as if the night itself was holding its breath. In that quiet pause, beneath the fading glow of streetlights, they simply existed—tired, content, and waiting for the dawn to carry them home.
- **Fine-tuned:** The two friends sat on the wall, their bodies slumped and weary, the night’s laughter still echoing faintly in their ears. The party had ended, and silence stretched between them, thick and heavy. They were too tired to call a taxi yet, each lost in the slow, drifting moment of exhaustion and quiet companionship. The stars above watched silently, and in that stillness, they found a fragile peace, waiting just a little longer before the journey home could begin.
---
- **Message 9:** Render this scene as short literary prose:\nA teacher stays in an empty classroom after the last lesson, looking at drawings the children have left on the wall.
- **Provided Answer:** After the final bell, the classroom changed quickly. Chairs stood crooked beneath the tables, pencil shavings brightened the floor, and the children's voices retreated along the corridor. She remained by the wall where their drawings had been pinned in two uneven rows. Houses leaned, suns occupied impossible corners, families smiled with identical arms. One picture had begun to curl away from its tack. She pressed it flat with her palm, then gathered her papers slowly, listening to the building settle around her.
- **Baseline Output:** The classroom was silent now, the last lesson long over. She lingered, her gaze drifting across the walls where children's drawings clung like quiet whispers of their imaginations. Crayon strokes and crumpled paper, bright with innocence and wonder, told stories only they knew. She reached out, fingers brushing a faded scribble, feeling the gentle weight of their small worlds left behind. In the stillness, she stayed a moment longer, holding onto the fleeting magic of their childhood, before turning away to leave the quiet room behind.
- **Fine-tuned:**  The teacher remained in the empty classroom long after the last lesson had ended, her gaze fixed on the drawings the children had left on the wall. The colors and shapes whispered stories only she could read, each one a fragile piece of the day’s fleeting magic. She stood there quietly, as if the walls themselves needed a moment to hold the memory, and when she finally turned to leave, the room seemed to breathe a little slower, waiting for the next time the children’s art would once again fill the space with life.
---
- **Message 10:** Create a literary vignette from this scene:\nA woman buys flowers for herself on the way home after a difficult day at work.
- **Provided Answer:** She chose the flowers without much thought: a small bunch wrapped in clear plastic, yellow and white among the vegetables and newspapers near the till. On the bus home she held them upright between her knees so the stems would not bend. The day had left a heaviness behind her eyes, though nothing dramatic had happened. In her kitchen she found a jar because the vase was dusty. Once the flowers were in water, the room looked only slightly different. She noticed the difference anyway.
- **Baseline Output:** She paused at the flower stall, the scent of fresh blooms wrapping around her like a quiet reassurance. With deliberate care, she chose a bouquet—vivid petals and delicate greens—something to remind her of beauty amid the chaos. As she paid, her fingers brushed the cool paper wrapping, a small act of kindness she granted herself. Walking home, the flowers nestled in her arms, she felt a subtle lift, a gentle reminder that even on the hardest days, there was still a moment worth savoring—something simple, yet enough to make her smile.
- **Fine-tuned:** The woman stopped at the flower stall on her way home, her hands trembling slightly as she chose the brightest blooms. She bought the flowers for herself, a small act of rebellion against the weight of the day. Once home, she placed them on the table, their colors a burst of life in the quiet room. As she watched the petals unfurl, she felt a slow, steady easing—an unspoken promise that even the hardest days could be softened by beauty, and that sometimes, the only person who needed to be cared for was herself.

## 7.1 Audit the training targets after evaluation

This post-run diagnostic checks whether the gold-standard answers used for training actually demonstrate the behaviour expected in evaluation. It does not replace the held-out comparison and does not use the test outputs to change the training result. Each training answer is scored against its own input using the same five binary criteria from Section 7.


| Training ID | Score /5 | Audit note |
|---|---:|---|
| train_01 | 5 | Preserves the delayed train and unopened letter while using concrete station detail and indirect tension. |
| train_02 | 5 | Preserves the closing bookshop scene with sensory detail and restrained implication. |
| train_03 | 5 | Preserves the sisters' disagreement while showing it through action rather than explanation. |
| train_04 | 5 | Preserves the waiting phone call and expresses anticipation through concrete kitchen actions. |
| train_05 | 5 | Preserves the lost toy and uses specific physical detail without adding a resolution. |
| train_06 | 5 | Preserves the unchanged but unfamiliar apartment and implies dislocation through setting. |
| train_07 | 5 | Preserves the research-placement wait and conveys tension through gesture and environment. |
| train_08 | 5 | Preserves the moving-van scene and creates atmosphere through observable detail. |
| train_09 | 5 | Preserves the early arrival and host disruption with restrained, scene-specific prose. |
| train_10 | 4 | Strong factual preservation, atmosphere, register, and discipline; loses one point because it explicitly names "embarrassment" rather than only implying emotion. |
| **Total** | **49 / 50** | **Nine targets score 5/5; one scores 4/5.** |

**Interpretation:** The training targets are consistently strong examples of the intended transformation, so the held-out tie of 33/50 does not point primarily to weak gold-standard writing. The more plausible limitation is the very small training set: 10 examples are not enough to reliably teach the model when to preserve concrete facts, imply emotion, meet the word range, and avoid generic literary language across varied scenes. A follow-up should retain this target quality while adding substantially more diverse, manually edited examples.

## 8. Conclusion



Using `gpt-4.1-nano-2025-04-14` and the same system message and generation settings, the selected step-112 checkpoint scored **33/50**, equal to the base model's **33/50**, across 10 held-out prompts. SFT therefore did not improve consistent literary transformation under the same minimal prompt in this experiment. The checkpoint was somewhat more restrained on test_01 and test_02, but this was offset by a too-short, generic output on test_03, an invented door opening on test_04, and continued direct naming of emotion or generic phrasing in several outputs.

Meaning preservation and atmosphere were generally present in both models, so neither showed a clear advantage on those behaviours. The fine-tuned model did not consistently improve implied emotion, literary register, or output discipline: both models failed the 60-100-word requirement on some prompts, and the checkpoint added unsupported details in some cases. The equal held-out score means the observed improvement is not large enough to justify the additional dataset preparation, training cost, and model-management complexity for this version of the task.

A key limitation is the very small synthetic dataset of 10 training examples and 10 validation examples. Its near-perfect training diagnostics are likely optimistic and did not translate into a higher held-out behavioural score. A larger follow-up experiment should add more diverse, manually edited scene-to-vignette pairs, especially examples that reward concrete source details, indirect emotion, 60-100-word outputs, and avoidance of cliches or invented resolutions.


## References

- Handout 3, pp. 31-35 (final challenge and practitioner questions).
- OpenAI, [Supervised fine-tuning guide](https://developers.openai.com/api/docs/guides/supervised-fine-tuning). Check this on the day you work: account eligibility and available models can differ by organization.